# Read-Only SQL Tool Test Suite

Tests valid SELECT queries, sales table record counts, and strict rejection of dangerous operations (INSERT, UPDATE, DROP, multiple statements).

In [ ]:
import sys
from pathlib import Path

# Resolve project root
cwd = Path.cwd().resolve()
if cwd.name == "tests":
    project_root = cwd.parent.parent
elif cwd.name == "backend":
    project_root = cwd.parent
else:
    project_root = cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.app.tools.sql_tool import execute_sql_query, validate_read_only_sql
print("SQL Tool successfully imported!")

In [ ]:
# Test 1: Valid simple SELECT query
res1 = execute_sql_query("SELECT 1 AS status, 'Hello SQL Tool' AS msg;")
print("Test 1 - Valid SELECT:", res1)
assert res1["success"] is True
assert res1["rows"][0]["status"] == 1

In [ ]:
# Test 2: Valid SELECT COUNT(*) FROM sales;
res2 = execute_sql_query("SELECT COUNT(*) AS total_sales FROM sales;")
print("Test 2 - Sales Count:", res2)
assert res2["success"] is True
assert res2["rows"][0]["total_sales"] > 0

In [ ]:
# Test 3: INSERT Rejection
res3 = execute_sql_query("INSERT INTO sales (amount) VALUES (100.0);")
print("Test 3 - INSERT Rejection:", res3)
assert res3["success"] is False
assert "Forbidden SQL operation detected: INSERT" in res3["error"]

In [ ]:
# Test 4: UPDATE Rejection
res4 = execute_sql_query("UPDATE sales SET amount = 200.0;")
print("Test 4 - UPDATE Rejection:", res4)
assert res4["success"] is False
assert "Forbidden SQL operation detected: UPDATE" in res4["error"]

In [ ]:
# Test 5: DROP Rejection
res5 = execute_sql_query("DROP TABLE sales;")
print("Test 5 - DROP Rejection:", res5)
assert res5["success"] is False
assert "Forbidden SQL operation detected: DROP" in res5["error"]

In [ ]:
# Test 6: Multiple Statements Rejection
res6 = execute_sql_query("SELECT 1; SELECT 2;")
print("Test 6 - Multiple Statements Rejection:", res6)
assert res6["success"] is False
assert "Multiple SQL statements are not allowed" in res6["error"]